In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error,mean_squared_error
import warnings

In [ ]:
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12,6)

In [ ]:
# Load the cleaned resume_JD pairs dataset
df=pd.read_csv('/content/cleaned_resumeJD_pairs.csv')

In [ ]:
df.head()

In [ ]:
print(f'loaded: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')
print("\nLabel distribution: ")
print(df['match_label'].value_counts())

In [ ]:
print('Loading the bert model...')
model= SentenceTransformer('all-mpnet-base-v2')

print('Model loaded!')
print(f"Model: all-mpnet-base-v2")
print(f" Parameters: 22.7M")
print(f" Embed dims: 768")
print(f" Max tokens: {model.max_seq_length}")

In [ ]:
# Part 2: Understanding Embeddings

texts = [
    'Python Developers with Django experience',
    'Python Software Engineer, Django REST APIs',
    'Fronted Developer React and javaScript',
    'Executive Chef with Michelin experience',
    'Data Scientist, Machine Learning, PyTorch',
]
embeddings = model.encode(texts)
sim_matrix = cosine_similarity(embeddings)



In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(sim_matrix, annot=True, fmt=".2f", cmap='RdYlGn',
            xticklabels=[t[:25] for t in texts],
            yticklabels=[t[:25] for t in texts],
            vmin=0, vmax=1
            )

plt.title('Semantic Similarity -BERT Embeddings', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Generating resume and job description embeddings
# We embed BOTH resume and job description separately

resume_embeddings= model.encode(
    df['resume_text'].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

In [ ]:
# Job description embeddings
job_embeddings=model.encode(
    df['job_description'].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

In [ ]:
print(f'\nResume embeddings shape: {resume_embeddings.shape}')
print(f'JD embeddings shape: {job_embeddings.shape}')
print(f"Each text => {resume_embeddings.shape[1]} - dimensional vector")

In [ ]:
resume_embeddings

In [ ]:
# compute cosine similarity for each pair

pair_similarities=[]

for i in range(len(resume_embeddings)):
   sim=cosine_similarity([resume_embeddings[i]], [job_embeddings[i]])[0][0]
   pair_similarities.append(sim)

df['bert_similarity']= pair_similarities

print("Cosine similarity computed for all pairs.")
print(df[['match_label', 'match_score', 'bert_similarity']].head(10))

In [ ]:
# Base Model Performance: How is BERT out of the box?

mae=mean_absolute_error(df['match_score'], df['bert_similarity'])
rmse= np.sqrt(mean_squared_error(df['match_score'], df['bert_similarity']))

print('Base BERT Model performance (before fine-tuning)')
print(f' MAE: {mae:.4f}   (lower is better)')
print(f' RMSE: {rmse:.4f}  (lower is better)')


In [ ]:
# Scatter Plot - predicted vs ground truth

plt.figure(figsize=(8,6))
colors=df['match_label'].map({'low': 'red', 'medium': 'orange', 'high': 'green'})
plt.scatter(df['match_score'], df['bert_similarity'], c=colors, alpha=0.6)
plt.plot([0, 1], [0, 1], 'k--', label='Perfect prediction')

plt.xlabel("Ground Truth Match Score")
plt.ylabel("Predicted Match Score")
plt.title("Predicted vs Ground Truth Match Score")

#Legend
from matplotlib.patches import Patch
plt.legend(handles=[
    Patch(facecolor='red', label='Low Match Label'),
    Patch(facecolor='orange', label='Medium Match Label'),
    Patch(facecolor='green', label='High Match Label'),
    plt.Line2D([0], [0], color='black', linestyle='--', label="Perfect")
])
plt.tight_layout()
plt.show()

In [ ]:
# Similarity score by label

fig, axes= plt.subplots(1, 2, figsize=(14,5))

# Histogram : BERT similarity grouped by label
for label, color in [('low','red'), ('medium', 'orange'),('high', 'green')]:
  subset= df[df['match_label'] == label]['bert_similarity']
  axes[0].hist(subset, bins=20, alpha=0.6, label=label, color=color)

axes[0].set_xlabel('BERT Similarity')
axes[0].set_ylabel('Frequency')
axes[0].set_title('BERT Similarity by Label')
axes[0].legend()

In [ ]:
# Similarity score by label

fig, axes= plt.subplots(1, 2, figsize=(14,5))

# Histogram : BERT similarity grouped by label
for label, color in [('low','red'), ('medium', 'orange'),('high', 'green')]:
  subset= df[df['match_label'] == label]['bert_similarity']
  axes[0].hist(subset, bins=20, alpha=0.6, label=label, color=color)

axes[0].set_xlabel('BERT Similarity')
axes[0].set_ylabel('Frequency')
axes[0].set_title('BERT Similarity by Label')
axes[0].legend()

# Box plot
df.boxplot(column='bert_similarity', by='match_label', ax=axes[1])
axes[1].set_xlabel('Match Label')
axes[1].set_ylabel('BERT Similarity')
axes[1].set_title('BERT Similarity by Match Label')

plt.tight_layout()
plt.show()

In [ ]:
# Store embeddings and store in our system
import pickle

with open('resume_embeddings.pkl', 'wb') as f:
  pickle.dump({
      'resume_embeddings': resume_embeddings,
      'job_embeddings': job_embeddings,
      'match_scores': df['match_score'].tolist(),
      'match_labels': df['match_label'].tolist(),
  }, f)

print("Saved: resume_embeddings.pkl")
print(f'Resume embeddings shape: {resume_embeddings.shape}')
print(f'JD embeddings shape: {job_embeddings.shape}')

print('\n these will be loaded for fine tuning')

In [ ]:
# Store embeddings and store in our system
import pickle

with open('resume_embeddings2.pkl', 'wb') as f:
  pickle.dump({
      'resume_embeddings': resume_embeddings,
      'job_embeddings': job_embeddings,
      'match_scores': df['match_score'].tolist(),
      'match_labels': df['match_label'].tolist(),
  }, f)

print("Saved: resume_embeddings.pkl")
print(f'Resume embeddings shape: {resume_embeddings.shape}')
print(f'JD embeddings shape: {job_embeddings.shape}')

print('\n these will be loaded for fine tuning')